In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import zipfile
import os
#https://www.kaggle.com/datasets/pkdarabi/vehicle-detection-image-dataset?resource=download
zip_file = '/content/drive/MyDrive/FormacaoAcademica/VisãoComputacional/TrabalhoDOIV/Find3dprint.coco.zip'
output_dir = '/content/label'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
  zip_ref.extractall(output_dir)

print("File unzipped!")

File unzipped!


In [3]:
import zipfile
import os
#https://www.kaggle.com/datasets/pkdarabi/vehicle-detection-image-dataset?resource=download
zip_file = '/content/drive/MyDrive/FormacaoAcademica/VisãoComputacional/TrabalhoDOIV/Teste.v2i.coco.zip'
output_dir = '/content/label2'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
  zip_ref.extractall(output_dir)

print("File unzipped!")

File unzipped!


In [4]:

#!pip3 install git+https://github.com/labelmeai/labelme

In [5]:
# install dependencies
!pip install -U torch torchvision cython
!pip install -U 'git+https://github.com/facebookresearch/fvcore.git' 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
import torch, torchvision
torch.__version__

  Cloning https://github.com/facebookresearch/fvcore.git to /tmp/pip-req-build-9whwqmpm
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/fvcore.git /tmp/pip-req-build-9whwqmpm
  Resolved https://github.com/facebookresearch/fvcore.git to commit 4882064e1dce905fd2211cf6971d238e45c98843
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-req-build-6dz8v0eu
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-req-build-6dz8v0eu
  Resolved https://github.com/cocodataset/cocoapi.git to commit 8c9bcc3cf640524c4c20a9c40e89cb6a2f2fa0e9
  Preparing metadata (setup.py) ... done


'2.12.1+cu130'

In [9]:
!git clone https://github.com/facebookresearch/detectron2 detectron2_repo
!pip install -e detectron2_repo

Cloning into 'detectron2_repo'...
remote: Enumerating objects: 16028, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 16028 (delta 19), reused 5 (delta 5), pack-reused 15995 (from 2)
Receiving objects: 100% (16028/16028), 6.80 MiB | 21.23 MiB/s, done.
Resolving deltas: 100% (11369/11369), done.
Obtaining file:///content/detectron2_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for detectron2
    error: subprocess-exited-with-error
    
    × python setup.py develop did not run successfully.
    │ exit code: 1
    ╰─> See above for output.
    
    note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× python setup.py develop did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [8]:
# You may need to restart your runtime prior to this, to let your installation take effect
# Some basic setup
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import cv2
import random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

ModuleNotFoundError: No module named 'detectron2'

In [ ]:
#if your dataset is in COCO format, this cell can be replaced by the following three lines:


from detectron2.data.datasets import register_coco_instances

register_coco_instances("my_dataset_train", {},"/content/label2/t_annotations.coco.json", "/content/label2/train")
register_coco_instances("my_dataset_val", {},"/content/label2/v_annotations.coco.json", "/content/label2/valid")


In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer
from detectron2.engine import DefaultTrainer

# Carregar o dataset
dataset_dicts = DatasetCatalog.get("my_dataset_train")
metadata = MetadataCatalog.get("my_dataset_train")

# Selecionar amostras aleatórias
sample_count = 5  # Número de amostras a serem selecionadas
random_samples = random.sample(dataset_dicts, sample_count)

# Função para plotar as imagens com as bounding boxes
def plot_samples_with_bboxes(samples, metadata):
    for sample in samples:
        # Carregar a imagem
        img = cv2.imread(sample["file_name"])
        # Visualizador para as bounding boxes
        visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, scale=1.2)
        vis = visualizer.draw_dataset_dict(sample)

        # Exibir a imagem com as bounding boxes
        plt.figure(figsize=(10,10))
        plt.imshow(vis.get_image()[:, :, ::-1])
        plt.axis("off")
        plt.show()

# Plotar as amostras selecionadas com as bounding boxes
plot_samples_with_bboxes(random_samples, metadata)


In [ ]:
from detectron2 import model_zoo

# Exemplo de caminho para o arquivo de configuração de um modelo de segmentação (Mask R-CNN)
config_path = model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
print(config_path)  # Exibe o caminho completo do arquivo de configuração


In [ ]:
import json

# Caminho do arquivo de anotações
file_path = "/content/label/t_annotations.coco.json"

# Carregar o arquivo JSON
with open(file_path, 'r') as file:
    coco_data = json.load(file)

# Extrair as categorias (classes) e seus IDs e nomes
categories = coco_data["categories"]
category_info = [(category["id"], category["name"]) for category in categories]
category_ids = {category["id"] for category in categories}
num_classes = len(categories)

# Exibir o total de classes, IDs das categorias e nomes das classes
print(f"Total de classes: {num_classes}")
print(f"IDs das categorias: {category_ids}")
print("Nomes das categorias e seus respectivos IDs:")
for category_id, category_name in category_info:
    print(f"ID: {category_id}, Nome: {category_name}")



In [ ]:
import json

# Carregar o arquivo de anotações
file_path = "/content/label/t_annotations.coco.json"
with open(file_path, 'r') as file:
    coco_data = json.load(file)

# Verificar IDs de classes
category_ids = {category['id'] for category in coco_data['categories']}
annotation_category_ids = {annotation['category_id'] for annotation in coco_data['annotations']}

# IDs de categorias ausentes nas anotações
missing_category_ids = annotation_category_ids - category_ids
if missing_category_ids:
    print(f"IDs de categorias ausentes: {missing_category_ids}")

# Verificar as bounding boxes
for annotation in coco_data['annotations']:
    bbox = annotation['bbox']
    if bbox[2] <= 0 or bbox[3] <= 0:
        print(f"Bounding box inválida: {bbox}")


In [ ]:
category_ids

In [ ]:
import torch
print(torch.cuda.is_available())  # Isso deve retornar True se a GPU estiver disponível


In [ ]:
import detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.utils.visualizer import Visualizer
import random
import os
from detectron2.data import build_detection_test_loader


# Configuração do modelo
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 6  # Ajuste dependendo da memória da GPU
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4  # Número de classes no dataset, excluindo fundo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml")  # Usando pesos pré-treinados
cfg.SOLVER.IMS_PER_BATCH = 2  # Ajuste dependendo da memória da GPU
cfg.SOLVER.BASE_LR = 0.00025  # Taxa de aprendizado
cfg.SOLVER.MAX_ITER = 300  # Número de iterações
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Limite para predições durante a validação/teste
cfg.MODEL.DEVICE = "cuda"  # Certifique-se de que a GPU está sendo utilizada

# Criar diretório de saída para salvar os resultados
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Treinamento do modelo
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

# Avaliar o modelo após o treinamento
evaluator = COCOEvaluator("my_dataset_val", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "my_dataset_val")
inference_on_dataset(trainer.model, val_loader, evaluator)

In [ ]:

# Avaliar o modelo após o treinamento
evaluator = COCOEvaluator("my_dataset_val", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "my_dataset_val")
inference_on_dataset(trainer.model, val_loader, evaluator)

In [ ]:
import torch
import random
import matplotlib.pyplot as plt
from detectron2.engine import DefaultPredictor
from detectron2.data import DatasetCatalog
from detectron2.utils.visualizer import Visualizer
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator
from detectron2.config import get_cfg
from detectron2 import model_zoo

# Configuração do modelo
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_val",)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4  # Número de classes no dataset, excluindo fundo
cfg.MODEL.WEIGHTS = "/content/output/model_final.pth"  # Caminho para o modelo treinado
cfg.MODEL.DEVICE = "cuda"  # Certifique-se de que a GPU está sendo utilizada
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.60  # Limite para predições durante a validação/teste

# Inicializar o preditor
predictor = DefaultPredictor(cfg)

# Carregar as imagens da validação
dataset_dicts = DatasetCatalog.get("my_dataset_val")

# Selecionar algumas imagens aleatórias para realizar a inferência
num_images = 4  # Número de imagens para inferência
sample_images = random.sample(dataset_dicts, num_images)

# Realizar a inferência e plotar as previsões
for img_dict in sample_images:
    img_path = img_dict["file_name"]
    # Carregar a imagem
    img = plt.imread(img_path)

    # Realizar a inferência
    outputs = predictor(img)
    #print(outputs)
    # Visualizar as previsões
    visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("my_dataset_val"), scale=1.2)
    vis = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))

    # Exibir a imagem com as previsões
    plt.figure(figsize=(10, 10))
    plt.imshow(vis.get_image()[:, :, ::-1])
    plt.axis("off")
    plt.show()
